# Week 11 Lab: Ensembles

Three short exercises on this week's ensemble methods: **voting**, **bagging**, and **random-forest feature
importance**. Each gives you most of the code; you fill in the model and interpret the results.

**Core**: Exercises 1–3 (about 40 minutes). **Extension**: Exercises 4–5 (optional).

Run this first: it imports everything the Core exercises use.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, BaggingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# scikit-learn 1.9 warns that SVC(probability=True) will change in a future version; we can ignore it
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

#### **Exercise 1**

**Voting.** Compare three classifiers with a hard-voting and a soft-voting ensemble of the same three.
`SVC` (a support vector machine) and `KNeighborsClassifier` are classifiers we don't cover in detail (KNN
comes in week 12). Treat them as black boxes: like every scikit-learn classifier, they have `fit`,
`predict`, and `score`.

In [ ]:
X, y = make_classification(n_samples=1000, n_features=20, n_informative=15,
                           n_redundant=5, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Base classifiers
log_clf = LogisticRegression(random_state=42)
svc_clf = SVC(probability=True, random_state=42)   # probability=True is needed for soft voting
knn_clf = KNeighborsClassifier()

# YOUR CODE HERE: create voting_hard and voting_soft, two VotingClassifiers over the three
# base classifiers (voting='hard' and voting='soft')
voting_hard = None
voting_soft = None

# YOUR CODE HERE: fit each of the five classifiers on the training data and store its
# test accuracy in results, e.g. results['Hard voting'] = 0.91
results = {}

pd.Series(results).round(3)

**Questions**

1. Which voting method did better here? Why might that be? (Hint: what extra information does soft voting use?)
2. How did the ensembles compare with the individual classifiers?

_Answer here_

#### **Exercise 2**

**Bagging.** On the breast cancer data, see how the number of trees and the fraction of the training data
each tree gets (`max_samples`) affect a bagging ensemble. Complete `evaluate_bagging` and the loop that fills
the two score arrays; the heatmaps are drawn for you. The first line printed is a single decision tree, for
comparison.

In [ ]:
cancer = load_breast_cancer()
X_c, y_c = cancer.data, cancer.target
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_c, y_c, test_size=0.2, random_state=42)

# For comparison: one unpruned decision tree
single = DecisionTreeClassifier(random_state=42).fit(X_train_c, y_train_c)
print(f"Single tree: train {single.score(X_train_c, y_train_c):.3f}, test {single.score(X_test_c, y_test_c):.3f}")


def evaluate_bagging(n_estimators, max_samples):
    """Fit a BaggingClassifier of decision trees; return (train accuracy, test accuracy)."""
    # YOUR CODE HERE: create a BaggingClassifier with estimator=DecisionTreeClassifier(),
    # the given n_estimators and max_samples, and random_state=42; fit it on the training data
    train_score, test_score = None, None
    return train_score, test_score

sample_sizes = [0.1, 0.3, 0.5, 0.7, 1.0]
n_estimators_list = [10, 50, 100, 200]
train_scores = np.zeros((len(sample_sizes), len(n_estimators_list)))
test_scores = np.zeros((len(sample_sizes), len(n_estimators_list)))

# YOUR CODE HERE: fill train_scores[i, j] and test_scores[i, j] for every combination
# of sample_sizes[i] and n_estimators_list[j]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, scores, title in [(axes[0], train_scores, 'Training accuracy'), (axes[1], test_scores, 'Test accuracy')]:
    im = ax.imshow(scores, cmap='YlOrRd', aspect='auto')
    for i in range(len(sample_sizes)):
        for j in range(len(n_estimators_list)):
            ax.text(j, i, f'{scores[i, j]:.3f}', ha='center', va='center')
    ax.set_xticks(range(len(n_estimators_list)), n_estimators_list)
    ax.set_yticks(range(len(sample_sizes)), sample_sizes)
    ax.set_xlabel('Number of trees')
    ax.set_ylabel('max_samples (fraction)')
    ax.set_title(title)
    fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

**Questions**

1. How does the number of trees affect test accuracy? Is there a point of diminishing returns?
2. How does `max_samples` affect the gap between training and test accuracy?
3. How does bagging compare with the single tree?

_Answer here_

#### **Exercise 3**

**Feature importance.** A random forest records how much each feature reduced impurity across all its
trees. Train one on the breast cancer data and plot its 10 most important features.

In [ ]:
rf_cancer = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train_c, y_train_c)
print('Random forest test accuracy:', round(rf_cancer.score(X_test_c, y_test_c), 3))

# YOUR CODE HERE: put rf_cancer.feature_importances_ in a pandas Series indexed by
# cancer.feature_names, and plot the 10 largest as a horizontal bar chart

**Questions**

1. Which features matter most? How much of the total importance do the top few hold?
2. How could you use these importances to select features for another model? (See `SelectFromModel` in the
   lecture.)

_Answer here_

---

### Extension exercises (optional)

#### **Exercise 4** (extension)

**Random forest parameters.** Use a `RandomForestRegressor` on the California housing data (the target is the
median house value, in $100,000s) and see how `max_depth` and `max_features` affect training and test error.
This one takes a minute or two to run.

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

housing = fetch_california_housing()   # downloads the data the first time (needs internet)
X_h, y_h = housing.data, housing.target
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(X_h, y_h, test_size=0.2, random_state=42)

# YOUR CODE HERE: for each max_depth in [5, 10, None] and each max_features in ['sqrt', None],
# fit RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42) and record train and
# test RMSE (root_mean_squared_error). Show the results as a table.

**Questions**

1. How does `max_depth` affect the gap between training and test error? What does that say about bias and
   variance?
2. What difference does `max_features` make?

_Answer here_

#### **Exercise 5** (extension)

**Out-of-bag evaluation vs. cross-validation.** Both estimate how a random forest will do on new data. Compare
their estimates, and how long each takes, as the number of trees grows.

In [ ]:
import time

n_estimators_range = [10, 50, 100, 200]

# YOUR CODE HERE: for each n_estimators, on the breast cancer data (X_c, y_c):
#   - fit RandomForestClassifier(n_estimators=..., oob_score=True, random_state=42) and record oob_score_
#   - compute the mean 5-fold cross_val_score of the same forest (without oob_score)
#   - time both (time.time() before and after)
# Show the results as a table.

**Questions**

1. How close are the OOB and cross-validation estimates?
2. Which is faster, and why?
3. When would you prefer each?

_Answer here_